# 🎵 Dia TTS for Google Colab

A complete Text-to-Speech system powered by the Dia model, optimized for Google Colab.

## Features
- **Voice Cloning**: Upload reference audio and clone voices
- **Predefined Voices**: Use curated voice samples
- **Multi-speaker Dialogue**: Generate conversations with [S1]/[S2] tags
- **GPU Acceleration**: Automatic hardware detection and optimization
- **Interactive UI**: Easy-to-use widgets for all parameters
- **Real-time Generation**: Live progress updates during audio generation

## Quick Start
1. Run the installation cell below
2. Load the TTS model
3. Enter your text and generate audio!

---

In [ ]:
#@title 🚀 Installation & Setup
#@markdown Run this cell first to install all required dependencies

import sys
import os
from pathlib import Path

# Check if we're in Google Colab
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("🔍 Running in Google Colab")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("🔍 Running in local environment")

# Install core dependencies
print("📦 Installing core dependencies...")
!pip install torch torchaudio numpy fastapi uvicorn -q
!pip install soundfile librosa pydub praat-parselmouth -q
!pip install huggingface_hub openai-whisper safetensors -q
!pip install ipywidgets matplotlib plotly -q
!pip install tqdm python-multipart pyyaml -q

# Install Dia model dependencies (if not already available)
try:
    import dia
    print("✅ Dia model package already installed")
except ImportError:
    print("📦 Installing Dia model package...")
    !pip install descript-audio-codec -q
    # Note: You may need to install the dia package separately
    # !pip install git+https://github.com/some-repo/dia.git -q

print("✅ Installation complete!")

# Create necessary directories
os.makedirs("/content/dia_tts/models", exist_ok=True)
os.makedirs("/content/dia_tts/reference_audio", exist_ok=True)
os.makedirs("/content/dia_tts/voices", exist_ok=True)
os.makedirs("/content/dia_tts/outputs", exist_ok=True)

print("📁 Created working directories:")
print("  /content/dia_tts/models")
print("  /content/dia_tts/reference_audio")
print("  /content/dia_tts/voices")
print("  /content/dia_tts/outputs")

In [ ]:
#@title 📚 Import Libraries & Modules
#@markdown Import all necessary modules for the TTS system

import sys
import os
import logging
from pathlib import Path

# Add current directory to path for imports
sys.path.append('/content')

# Import Colab-specific modules
try:
    from colab_config import config_manager, get_config, update_config, get_device
    from colab_files import get_reference_files, get_predefined_voices, save_uploaded_file
    from engine_colab import load_model, generate_speech, get_model_status
    from colab_ui import display_tts_interface, quick_generate
    print("✅ Successfully imported Colab TTS modules")
except ImportError as e:
    print(f"❌ Error importing modules: {e}")
    print("Make sure all files are in the current directory")

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)

# Reduce verbosity of some libraries
logging.getLogger('uvicorn.access').setLevel(logging.WARNING)
logging.getLogger('whisper').setLevel(logging.WARNING)
logging.getLogger('parselmouth').setLevel(logging.WARNING)

logger = logging.getLogger(__name__)
print("✅ Libraries imported successfully!")

In [ ]:
#@title 🤖 Load TTS Model
#@markdown Load the Dia TTS model (this may take several minutes on first run)

print("🔄 Loading Dia TTS model...")
print(f"📊 Hardware: {get_device()}")

# Load the model
success = load_model()

if success:
    model_status = get_model_status()
    print("✅ Model loaded successfully!")
    print(f"   Device: {model_status['device']}")
    print(f"   Sample Rate: {model_status['sample_rate']}Hz")
    print(f"   Model Type: {model_status['model_type']}")
    
    # Show GPU memory if using CUDA
    if model_status['device'] == 'cuda':
        import torch
        if torch.cuda.is_available():
            memory_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
            print(f"   GPU Memory: {memory_gb:.1f}GB")
else:
    print("❌ Failed to load model. Check the error messages above.")
    print("💡 Troubleshooting tips:")
    print("   - Make sure you have a stable internet connection")
    print("   - Try restarting the runtime if issues persist")
    print("   - Check that all dependencies are installed correctly")

In [ ]:
#@title 🎛️ Interactive TTS Interface
#@markdown Complete interface for text-to-speech generation

# Display the main TTS interface
display_tts_interface()

In [ ]:
#@title 🚀 Quick Examples
#@markdown Try these examples to test the system

# Example 1: Simple dialogue
print("🎭 Example 1: Simple Dialogue")
dialogue_text = "[S1] Hello there! How are you doing today? [S2] I'm doing great, thanks for asking! (laughs) [S1] That's wonderful to hear."
audio1 = quick_generate(dialogue_text, voice_mode="dialogue")
display(audio1)

# Example 2: Single speaker with emotion
print("🎭 Example 2: Single Speaker with Emotion")
emotion_text = "[S1] This is absolutely incredible! (excited) I can't believe how amazing this technology is. (laughs) Wow!"
audio2 = quick_generate(emotion_text, voice_mode="dialogue")
display(audio2)

# Example 3: Voice cloning (if you have reference files)
print("🎭 Example 3: Voice Cloning")
print("Note: This requires reference audio files in /content/dia_tts/reference_audio/")
ref_files = get_reference_files()
if ref_files:
    clone_text = "[S1] This is a test of voice cloning technology. I hope it works well!"
    audio3 = quick_generate(clone_text, voice_mode="clone", clone_reference_filename=ref_files[0])
    display(audio3)
else:
    print("No reference files found. Upload some audio files first!")

In [ ]:
#@title 📁 File Management
#@markdown Manage your reference audio and voice files

import ipywidgets as widgets
from IPython.display import display, HTML

# Display current files
print("📊 Current Files:")

ref_files = get_reference_files()
print(f"📁 Reference Audio Files ({len(ref_files)}):")
for file in ref_files:
    print(f"   • {file}")

predefined_voices = get_predefined_voices()
print(f"\n🎭 Predefined Voices ({len(predefined_voices)}):")
for voice in predefined_voices:
    print(f"   • {voice['display_name']} ({voice['filename']})")

# File upload widget
upload_widget = widgets.FileUpload(
    accept='.wav,.mp3,.txt',
    multiple=True,
    description='Upload Audio/Transcript Files'
)

def on_upload_change(change):
    if change['new']:
        for filename, file_info in change['new'].items():
            # Determine subfolder based on file type
            if filename.endswith('.txt'):
                subfolder = "reference_audio"
            else:
                subfolder = "reference_audio"
            
            file_path = save_uploaded_file(
                file_info['content'],
                filename,
                subfolder
            )
            print(f"✅ Uploaded: {filename}")
        
        # Refresh the file lists
        print("\n📊 Updated Files:")
        ref_files = get_reference_files()
        print(f"📁 Reference Audio Files ({len(ref_files)}):")
        for file in ref_files:
            print(f"   • {file}")

upload_widget.observe(on_upload_change, names='value')
display(widgets.HTML("<h4>📤 Upload Files:</h4>"))
display(upload_widget)

# Instructions
display(HTML("""
<div style="background-color: #e8f4fd; padding: 15px; border-radius: 5px; margin: 15px 0;">
<h4 style="color: #007bff; margin-top: 0;">📋 File Management Tips:</h4>
<ul>
<li><strong>Reference Audio:</strong> Upload .wav/.mp3 files for voice cloning</li>
<li><strong>Transcripts:</strong> Create .txt files with the same name as audio files containing the transcript</li>
<li><strong>Voice Cloning:</strong> For best results, use clear speech with matching transcripts</li>
<li><strong>Predefined Voices:</strong> Place .wav files in the voices folder for consistent voice generation</li>
</ul>
</div>
"""))

In [ ]:
#@title ⚙️ Advanced Settings
#@markdown Fine-tune the TTS system configuration

import ipywidgets as widgets
from IPython.display import display

# Get current config
config = get_config()

# Create configuration widgets
model_repo_input = widgets.Text(
    value=config.model_repo_id,
    description='Model Repository:',
    style={'description_width': 'initial'}
)

device_dropdown = widgets.Dropdown(
    options=['auto', 'cpu', 'cuda'],
    value=config.device,
    description='Device:',
    style={'description_width': 'initial'}
)

use_gpu_checkbox = widgets.Checkbox(
    value=config.use_gpu,
    description='Use GPU (if available)',
    style={'description_width': 'initial'}
)

split_text_checkbox = widgets.Checkbox(
    value=config.split_text,
    description='Enable Text Splitting',
    style={'description_width': 'initial'}
)

def update_config_handler(button):
    updates = {
        'model_repo_id': model_repo_input.value,
        'device': device_dropdown.value,
        'use_gpu': use_gpu_checkbox.value,
        'split_text': split_text_checkbox.value,
    }
    
    update_config(updates)
    print("✅ Configuration updated!")
    print(f"   Model Repository: {updates['model_repo_id']}")
    print(f"   Device: {updates['device']}")
    print(f"   Use GPU: {updates['use_gpu']}")
    print(f"   Split Text: {updates['split_text']}")

update_btn = widgets.Button(
    description='💾 Update Configuration',
    button_style='primary'
)
update_btn.on_click(update_config_handler)

# Display configuration interface
display(widgets.HTML("<h3>⚙️ System Configuration</h3>"))
display(widgets.VBox([
    model_repo_input,
    device_dropdown,
    use_gpu_checkbox,
    split_text_checkbox,
    update_btn
]))

# Show current status
model_status = get_model_status()
display(widgets.HTML(f"""
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; margin: 15px 0;">
<h4>📊 Current Status:</h4>
<ul>
<li><strong>Model Loaded:</strong> {'✅ Yes' if model_status['loaded'] else '❌ No'}</li>
<li><strong>Device:</strong> {model_status['device']}</li>
<li><strong>Sample Rate:</strong> {model_status['sample_rate']}Hz</li>
<li><strong>Model Type:</strong> {model_status['model_type']}</li>
</ul>
</div>
"""))

## 💡 Tips & Tricks for Dia TTS in Colab

### Text Formatting
- Use `[S1]` and `[S2]` tags for multi-speaker dialogue
- Add emotions: `(laughs)`, `(sighs)`, `(excited)`, etc.
- Use prosody controls: `[EMO=happy]`, `[RATE=0.8]`
- Add pauses: `(pause=0.5)`, `...` for thoughtful pauses

### Voice Cloning Best Practices
- Upload clear, high-quality reference audio (16kHz+ recommended)
- Create transcript files with the exact same name as audio files
- Use `[S1]` tags in transcripts for single-speaker references
- Keep reference audio under 20 seconds for best results

### Performance Optimization
- Enable GPU in Runtime settings for faster generation
- Use text splitting for long inputs (>200 characters)
- Lower temperature (0.8-1.0) for more consistent output
- Higher CFG Scale (2.5-4.0) for better prompt adherence

### Troubleshooting
- **Model won't load**: Check internet connection and disk space
- **Poor audio quality**: Try different generation parameters
- **Voice cloning issues**: Ensure transcript matches audio exactly
- **Out of memory**: Reduce chunk size or disable GPU

### File Organization
```
/content/dia_tts/
├── models/           # Downloaded model files
├── reference_audio/  # Voice cloning files (.wav/.mp3 + .txt)
├── voices/          # Predefined voice samples (.wav)
└── outputs/         # Generated audio files
```

---

## 🎯 Next Steps

1. **Upload Reference Audio**: Use the file upload widget to add voice samples
2. **Create Transcripts**: Make .txt files with the same names as your audio files
3. **Experiment**: Try different voice modes and parameters
4. **Generate**: Create amazing TTS content!

Happy synthesizing! 🎵